# MURA XR_WRIST — Minimal Agent (assembled from existing blocks)

**Status:** working draft for team review. Research use only — not a diagnosis; every study goes to human review.

**Flow:** declared anatomy + images → deterministic guard (`guarded_study_route`) → canonical Model B (mean of view probabilities, frozen threshold) → policy (`agent_policy.py` v1.1) → **only for priority studies:** TTA + Grad-CAM evidence → one CrewAI agent (Gemini) explains it through policy-gated read-only tools → post-checks → human review.

| Block | Source |
|---|---|
| QC, anatomy scope, view evidence, route | `src/runtime/deterministic_safety.py` (canonical) |
| Validation and tool selection | `tools/agent_policy.py` v1.1 (Hagit) |
| Model B weights, threshold, mean aggregation | `configs/model_b_artifact_bundle.json` + Drive `MODEL_B_FINE_TUNING` (canonical) |
| Preprocessing | `src/data/transforms.py::build_eval_transform` (canonical) |
| TTA, Grad-CAM | Notebook 09 (Hagit), Steps 19, 21, 25 — same logic, explicit dependencies |
| Policy-gated read-only tools | Notebook 09 (Hagit), Step 59 |
| Agent, task, Gemini settings | Notebook 09 (Hagit), Steps 93–94 |

**Rules enforced in code:** no official MURA `valid` paths; demo studies come from internal validation and are chosen by model output only (labels are never read); anonymised case IDs; the LLM cannot change the prediction, threshold or route; the LLM runs only when the policy selects tools; at most `MAX_LLM_CASES` LLM runs per session.

**Before running:** put the GitHub ZIP of `main` in `/content` (or set `MURA_PROJECT_ROOT`); add `GOOGLE_API_KEY` to Colab Secrets.

In [1]:
# 1. Agent dependencies (torch/torchvision are preinstalled in Colab)
%pip install -q crewai==1.15.36 google-genai==2.12.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.


In [2]:
# 2. Paths — found automatically (shallow search only); set an environment variable to override
import os, sys, io, json, copy, time, zipfile, importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE = Path("/content/drive")


def first_match(root, patterns):
    # First existing match, pattern by pattern; stops at the first hit and never walks the whole Drive.
    for pattern in patterns:
        match = next(root.glob(pattern), None)
        if match is not None:
            return match
    return None


# Data: MURA_DATA_DIR, else the team default MyDrive/DATA, else a DATA folder with Model B up to 2 levels deep.
if os.environ.get("MURA_DATA_DIR"):
    DATA_DIR = Path(os.environ["MURA_DATA_DIR"])
else:
    marker = first_match(DRIVE, ["MyDrive/DATA/MODEL_B_FINE_TUNING",
                                 "MyDrive/*/DATA/MODEL_B_FINE_TUNING",
                                 "MyDrive/*/*/DATA/MODEL_B_FINE_TUNING",
                                 ".shortcut-targets-by-id/*/*/*/DATA/MODEL_B_FINE_TUNING"])
    DATA_DIR = marker.parent if marker else DRIVE / "MyDrive" / "DATA"
MODEL_B_DIR = DATA_DIR / "MODEL_B_FINE_TUNING"
MURA_ROOT = DATA_DIR                      # images: DATA/train/XR_*/...
VAL_MANIFEST = Path(os.environ.get("MURA_INTERNAL_VAL_MANIFEST",
                                   str(DATA_DIR / "WRIST_SPLITS" / "wrist_internal_val.csv")))

# Code: MURA_PROJECT_ROOT, else unpack the GitHub ZIP of `main` from /content, inside DATA (shared copy), next to DATA, or the My Drive root.
PROJECT_ROOT = Path(os.environ.get("MURA_PROJECT_ROOT", "/content/mura-wrist-ml-pipeline-main"))
if not PROJECT_ROOT.exists():
    zip_name = "mura-wrist-ml-pipeline-main*.zip"
    REPO_ZIP = (Path(os.environ["MURA_REPO_ZIP"]) if os.environ.get("MURA_REPO_ZIP")
                else first_match(Path("/content"), [zip_name])
                or first_match(DATA_DIR, [zip_name])
                or first_match(DATA_DIR.parent, [zip_name])
                or first_match(DRIVE / "MyDrive", [zip_name]))
    if REPO_ZIP and REPO_ZIP.exists():
        with zipfile.ZipFile(REPO_ZIP) as archive:
            archive.extractall(PROJECT_ROOT.parent)
        print("Repository unpacked from:", REPO_ZIP)

LOCAL_MODEL_DIR = Path(os.environ.get("MURA_LOCAL_MODEL_DIR", "/content/model_b"))  # shared Drive is never written
OUTPUT_DIR = Path(os.environ.get("MURA_AGENT_OUTPUT_DIR",   # shared project folder; each run adds a new file
    str(DATA_DIR / "AGENT_OUTPUTS")))

print("DATA_DIR:", DATA_DIR)
for path in (PROJECT_ROOT / "src", MODEL_B_DIR / "checkpoint_parts", MURA_ROOT / "train" / "XR_WRIST", VAL_MANIFEST):
    print("OK     " if path.exists() else "MISSING", path)

Mounted at /content/drive
Repository unpacked from: /content/drive/MyDrive/DATA/mura-wrist-ml-pipeline-main.zip
DATA_DIR: /content/drive/MyDrive/DATA
OK      /content/mura-wrist-ml-pipeline-main/src
OK      /content/drive/MyDrive/DATA/MODEL_B_FINE_TUNING/checkpoint_parts
OK      /content/drive/MyDrive/DATA/train/XR_WRIST
OK      /content/drive/MyDrive/DATA/WRIST_SPLITS/wrist_internal_val.csv


In [3]:
# 3. Canonical project modules and the official-valid guard
sys.path.insert(0, str(PROJECT_ROOT))


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


from src.runtime.deterministic_safety import StudyImage, guarded_study_route
from src.models.resnet18_classifier import build_resnet18_classifier
from src.data.transforms import build_eval_transform

agent_policy = load_module("agent_policy", PROJECT_ROOT / "tools" / "agent_policy.py")
checkpoint_tool = load_module("reassemble_model_b_checkpoint",
                              PROJECT_ROOT / "scripts" / "reassemble_model_b_checkpoint.py")
assert agent_policy.POLICY_VERSION == "1.1"


def assert_not_official_valid(path):
    # Official MURA `valid` is reserved; this notebook never reads it.
    parts = {part.lower() for part in Path(path).parts}
    if parts & {"valid", "mura_official_validation"}:
        raise PermissionError(f"Official MURA valid is not allowed here: {path}")


print("Policy version:", agent_policy.POLICY_VERSION)

Policy version: 1.1


In [4]:
# 4. Canonical Model B: reassemble from Drive parts, verify SHA-256, load
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageEnhance

BUNDLE = json.loads((PROJECT_ROOT / "configs" / "model_b_artifact_bundle.json").read_text())
THRESHOLD = float(BUNDLE["model"]["threshold"])
EXPECTED_SHA256 = BUNDLE["required_files"]["best_resnet18_study_level.pt"]["sha256"].upper()
assert BUNDLE["official_mura_valid_used_for_selection"] is False
assert BUNDLE["model"]["study_aggregation"] == "mean probability"

frozen_policy = json.loads((MODEL_B_DIR / "frozen_primary_operating_policy.json").read_text())
assert abs(float(frozen_policy["threshold"]) - THRESHOLD) < 1e-12, "Drive policy and repo bundle disagree"

artifact = checkpoint_tool.reassemble_checkpoint(MODEL_B_DIR / "checkpoint_parts", LOCAL_MODEL_DIR)
assert artifact["checkpoint_sha256"] == EXPECTED_SHA256, "Checkpoint is not the canonical Model B"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
try:
    checkpoint = torch.load(artifact["checkpoint"], map_location=DEVICE, weights_only=True)
except Exception:
    # SHA-256 was verified above, so a full unpickle of this file is acceptable.
    checkpoint = torch.load(artifact["checkpoint"], map_location=DEVICE, weights_only=False)
model = build_resnet18_classifier(weights="none")
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE).eval()
transform = build_eval_transform(image_size=224)

MODEL_ID = {"bundle_id": BUNDLE["bundle_id"], "checkpoint_sha256": EXPECTED_SHA256,
            "threshold": THRESHOLD, "aggregation": "mean"}
print(artifact["status"], "|", MODEL_ID)

reassembled_and_verified | {'bundle_id': 'checkpoint1-model-b-mean-probability-95-sensitivity', 'checkpoint_sha256': '63AEDC6BEC1534BDED7C3CA4D0DAF03AFC17BF92166CB7848506A6F3A82DE5B6', 'threshold': 0.263567715883255, 'aggregation': 'mean'}


/content/mura-wrist-ml-pipeline-main/src/models/resnet18_classifier.py:197: RuntimeWarning: Building a randomly initialized ResNet-18. This is suitable for a smoke test but not the required pretrained Phase 1 experiment.
  backbone, build_info = build_resnet18_backbone(


In [5]:
# 5. Model B view probabilities and the record checked by agent_policy.py
def view_probabilities(images):
    # Canonical preprocessing -> Model B -> one sigmoid probability per view.
    batch = torch.stack([transform(Image.open(io.BytesIO(image.image_bytes))) for image in images])
    with torch.inference_mode():
        return torch.sigmoid(model(batch.to(DEVICE))).cpu().tolist()


ACTION_BY_ROUTE = {"RAISE_PRIORITY": "priority_human_review", "STANDARD_HUMAN_REVIEW": "routine_workflow"}


def to_policy_record(case_id, route, probabilities):
    # Translate the canonical route into the record format validated by agent_policy.py.
    evidence = route["view_evidence"]
    return {
        "study_id": case_id,
        "workflow_status": "SUCCESS",
        "qc_status": "PASS" if all(q["status"] == "PASS" for q in route["qc_results"]) else "WARNING",
        "anatomy_status": route["anatomy"]["status"],
        "classification_status": "SUCCESS",
        "classification_performed": True,
        "classification_valid": True,
        "classification_modified": False,
        "predicted_class": int(route["model_predicted_abnormal"]),
        "study_probability": float(evidence["mean_probability"]),
        "classification_threshold": THRESHOLD,
        "agent_action": ACTION_BY_ROUTE[route["deterministic_outcome"]],
        "image_predictions": [float(p) for p in probabilities],
        "view_consistency": {
            "status": "SUCCESS",
            "image_count": int(evidence["view_count"]),
            "min_probability": float(evidence["min_probability"]),
            "max_probability": float(evidence["max_probability"]),
            "mean_probability": float(evidence["mean_probability"]),
            "probability_range": float(evidence["probability_range"]),
            "probability_std": float(evidence["probability_std"]),
            "threshold_disagreement": bool(evidence["threshold_disagreement"]),
            "image_classes": [int(p >= THRESHOLD) for p in probabilities],
            "threshold": THRESHOLD,
        },
        "model": MODEL_ID,
    }

In [6]:
# 6. Explanation evidence (Notebook 09, Steps 19, 21, 25 — same logic, explicit dependencies)
TTA_VARIANTS = {
    "original": lambda im: im,
    "brightness_plus_5": lambda im: ImageEnhance.Brightness(im).enhance(1.05),
    "brightness_minus_5": lambda im: ImageEnhance.Brightness(im).enhance(0.95),
    "contrast_plus_5": lambda im: ImageEnhance.Contrast(im).enhance(1.05),
    "contrast_minus_5": lambda im: ImageEnhance.Contrast(im).enhance(0.95),
}


def study_tta_summary(images):
    # Five deterministic variants per view; study probability per variant = mean over views.
    per_view = []
    for image in images:
        source = Image.open(io.BytesIO(image.image_bytes)).convert("RGB")
        batch = torch.stack([transform(variant(source)) for variant in TTA_VARIANTS.values()])
        with torch.inference_mode():
            per_view.append(dict(zip(TTA_VARIANTS, torch.sigmoid(model(batch.to(DEVICE))).cpu().tolist())))
    study = {name: float(np.mean([view[name] for view in per_view])) for name in TTA_VARIANTS}
    values = np.array(list(study.values()))
    crossing = bool((values >= THRESHOLD).any() and (values < THRESHOLD).any())
    return {
        "status": "SUCCESS",
        "study_probability_by_variant": study,
        "range": float(np.ptp(values)),
        "std": float(values.std()),
        "stability_category": "THRESHOLD_CROSSING" if crossing else "NO_THRESHOLD_CROSSING",
        "threshold": THRESHOLD,
        "classification_modified": False,
    }


GRADCAM_LAYER = model.backbone.layer4[-1].conv2


def gradcam_summary(image):
    # Grad-CAM for the abnormality logit. Only metadata leaves this function; no image is shown or saved.
    store = {}
    handle = GRADCAM_LAYER.register_forward_hook(lambda module, inputs, output: store.__setitem__("act", output))
    try:
        tensor = transform(Image.open(io.BytesIO(image.image_bytes)).convert("RGB")).unsqueeze(0).to(DEVICE)
        with torch.enable_grad():
            logit = model(tensor).reshape(-1)[0]
            grad = torch.autograd.grad(logit, store["act"])[0]
    finally:
        handle.remove()
    activation = store["act"].detach()
    cam = F.relu((grad.mean(dim=(2, 3), keepdim=True) * activation).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=tensor.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min()).clamp_min(1e-12)
    row, col = np.unravel_index(int(cam.argmax()), tuple(cam.shape))
    return {
        "image_id": image.image_id,
        "status": "SUCCESS",
        "target_layer": "layer4[-1].conv2",
        "view_probability": float(torch.sigmoid(logit.detach())),
        "heatmap_shape": list(cam.shape),
        "peak_relative_position_row_col": [round(float(row) / cam.shape[0], 2), round(float(col) / cam.shape[1], 2)],
        "classification_modified": False,
    }

In [7]:
# 7. Model card and policy-gated read-only tools (Notebook 09, Step 59 pattern)
from crewai.tools import tool

MODEL_CARD = {
    "model": BUNDLE["model"]["architecture"],
    "study_aggregation": BUNDLE["model"]["study_aggregation"],
    "operating_policy": BUNDLE["model"]["operating_policy"],
    "threshold": THRESHOLD,
    "internal_validation_sensitivity": round(BUNDLE["model"]["validation_sensitivity"], 3),
    "internal_validation_specificity": round(BUNDLE["model"]["validation_specificity"], 3),
    "selection_population": BUNDLE["selection_population"],
    "official_mura_valid_used_for_selection": False,
    "checkpoint_sha256": EXPECTED_SHA256,
    "intended_use": "Research decision support for XR_WRIST: prioritises human review; never a diagnosis.",
    "limitations": [
        "Low specificity at this operating point: many normal studies are flagged.",
        "Trained and validated on MURA only; no external validation.",
        "TTA and Grad-CAM describe model behaviour, not diagnostic correctness or lesion location.",
    ],
}


def make_gated_tools(record, evidence):
    # Tools return only pre-computed evidence, and only if the policy authorises it for this study.
    trusted_record, trusted_evidence = copy.deepcopy(record), copy.deepcopy(evidence)
    calls = []

    def authorised(study_id, policy_tool):
        if study_id != trusted_record["study_id"]:
            return False
        try:
            decision = agent_policy.select_agent_tools(trusted_record, THRESHOLD)
        except Exception:
            return False
        return decision["policy_status"] == "SUCCESS" and policy_tool in decision["selected_tools"]

    def answer(name, study_id, policy_tool, payload, limitation):
        allowed = policy_tool is None or authorised(study_id, policy_tool)
        calls.append({"tool": name, "study_id": study_id, "status": "SUCCESS" if allowed else "DENIED"})
        if not allowed:
            return json.dumps({"status": "DENIED", "reason": "Tool request not authorised."})
        return json.dumps({"status": "SUCCESS", "study_id": study_id, "results": payload,
                           "limitation": limitation, "requires_human_review": True}, allow_nan=False)

    @tool("mura_tta_summary")
    def mura_tta_summary(study_id: str) -> str:
        "Study-level test-time augmentation stability for an authorised MURA study."
        return answer("mura_tta_summary", study_id, "tta_uncertainty", trusted_evidence["tta"],
                      "TTA measures sensitivity to small image changes; it does not establish correctness.")

    @tool("mura_gradcam_summary")
    def mura_gradcam_summary(study_id: str) -> str:
        "Grad-CAM attribution metadata for each view of an authorised MURA study."
        return answer("mura_gradcam_summary", study_id, "gradcam_explainer", trusted_evidence["gradcam"],
                      "Grad-CAM shows where the model score is sensitive; it does not localise a fracture.")

    @tool("mura_model_card")
    def mura_model_card(study_id: str) -> str:
        "Frozen Model B card: intended use, operating point and limitations."
        return answer("mura_model_card", study_id, None, MODEL_CARD,
                      "Internal-validation figures; not an external evaluation.")

    return [mura_tta_summary, mura_gradcam_summary, mura_model_card], calls

In [8]:
# 8. One CrewAI agent with Gemini (Notebook 09, Steps 93–94). The key comes from Colab Secrets.
from crewai import Agent, Task, Crew, Process, LLM

GEMINI_MODEL = "gemini/gemini-3.1-flash-lite"
USD_PER_M_INPUT, USD_PER_M_OUTPUT = 0.25, 1.50   # as configured in Notebook 09 — check current Gemini pricing
MAX_LLM_CASES = 3                                # hard cap on LLM runs in this session

if IN_COLAB and not os.environ.get("GOOGLE_API_KEY"):
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
llm = LLM(model=GEMINI_MODEL, api_key=os.environ.get("GOOGLE_API_KEY"), temperature=0)

TASK_TEXT = (
    "Analyze MURA wrist study {study_id}.\n"
    "Frozen predicted class: {predicted_class} (1 = abnormal at the frozen threshold)\n"
    "Deterministic route: {route} (RAISE_PRIORITY = priority human review)\n"
    "Route reason (deterministic): {route_reason}\n"
    "Frozen study probability: {study_probability}\n"
    "Frozen threshold: {threshold}\n"
    "Views on both sides of the threshold: {view_disagreement}\n\n"
    "Call mura_tta_summary, mura_gradcam_summary and mura_model_card with the study identifier.\n"
    "State the route reason exactly as given; TTA, Grad-CAM and the model card are supplementary and did not decide the route.\n"
    "Summarize the tool evidence faithfully and briefly; do not add details the tools did not return.\n"
    "Do not change the model prediction. Do not infer the ground-truth label.\n"
    "State that human review is mandatory. Research use only; not a clinical diagnosis."
)


def build_crew(tools):
    agent = Agent(
        role="MURA Research Explanation Analyst",
        goal="Explain frozen MURA findings using authorized tools.",
        backstory=("You are a research assistant. Use verified TTA and Grad-CAM evidence "
                   "without changing the frozen model prediction."),
        tools=tools, llm=llm, allow_delegation=False, max_iter=4, max_retry_limit=0, verbose=False,
    )
    task = Task(
        description=TASK_TEXT,
        expected_output=("Short research report: frozen prediction, TTA evidence, Grad-CAM evidence, "
                         "model limitations and mandatory human review."),
        agent=agent,
    )
    return Crew(agents=[agent], tasks=[task], process=Process.sequential, memory=False, cache=False, verbose=False)

In [9]:
# 9. One study end to end
LLM_RUNS = 0


async def run_case(case_id, declared_anatomy, image_paths):
    global LLM_RUNS
    for path in image_paths:
        assert_not_official_valid(path)
    images = [StudyImage(Path(p).name, Path(p).read_bytes()) for p in image_paths]
    seen = {}

    def classifier(study_images):
        seen["p"] = view_probabilities(study_images)
        return seen["p"]

    # Guard before the model (canonical): anatomy, QC, view count, immutable threshold.
    route = guarded_study_route(case_id=case_id, declared_anatomy=declared_anatomy, images=images,
                                classifier=classifier, threshold=THRESHOLD)
    report = {"case_id": case_id, "declared_anatomy": declared_anatomy, "views": len(images),
              "route": route["deterministic_outcome"], "model": MODEL_ID,
              "policy_version": agent_policy.POLICY_VERSION, "human_review_required": True, "llm_called": False}
    if not route["classification_performed"]:
        report["reason"] = route["reason"]                      # blocked before the model
        return report

    # Policy: validates the record and selects tools.
    record = to_policy_record(case_id, route, seen["p"])
    decision = agent_policy.select_agent_tools(record, THRESHOLD)
    report.update(predicted_class=record["predicted_class"],
                  study_probability=round(record["study_probability"], 4),
                  view_disagreement=record["view_consistency"]["threshold_disagreement"],
                  policy_status=decision["policy_status"], selected_tools=decision["selected_tools"],
                  validation_issues=decision["validation_issues"])
    if decision["policy_status"] != "SUCCESS" or not decision["selected_tools"]:
        return report                                           # routine or safety block: no LLM

    evidence = {"tta": study_tta_summary(images), "gradcam": [gradcam_summary(image) for image in images]}
    report["evidence"] = evidence
    if LLM_RUNS >= MAX_LLM_CASES:
        report["llm_skipped"] = "MAX_LLM_CASES reached"
        return report

    tools, calls = make_gated_tools(record, evidence)
    snapshot = copy.deepcopy(record)
    LLM_RUNS += 1
    started = time.perf_counter()
    inputs = {
        "study_id": case_id,
        "route": route["deterministic_outcome"],
        "route_reason": "; ".join(reason for reason, active in (
            ("study probability at or above the threshold", record["predicted_class"] == 1),
            ("views fall on both sides of the threshold", record["view_consistency"]["threshold_disagreement"]),
        ) if active),
        "predicted_class": str(record["predicted_class"]),
        "study_probability": f"{record['study_probability']:.4f}",
        "threshold": f"{THRESHOLD:.6f}",
        "view_disagreement": str(record["view_consistency"]["threshold_disagreement"]),
    }
    try:
        output = await build_crew(tools).kickoff_async(inputs=inputs)
    except Exception as error:                              # API, billing or network failure: the deterministic route stands
        report.update(llm_error=f"{type(error).__name__}: {str(error)[:300]}",
                      duration_s=round(time.perf_counter() - started, 2))
        return report
    usage, text = output.token_usage, str(output.raw)
    report.update(
        llm_called=True,
        llm_model=GEMINI_MODEL,
        duration_s=round(time.perf_counter() - started, 2),
        tool_calls=calls,
        tokens={"input": usage.prompt_tokens, "output": usage.completion_tokens,
                "requests": usage.successful_requests},
        estimated_cost_usd=round((usage.prompt_tokens * USD_PER_M_INPUT
                                  + usage.completion_tokens * USD_PER_M_OUTPUT) / 1e6, 6),
        agent_report=text,
        post_checks={                                           # guard after the LLM
            "record_unchanged": record == snapshot,
            "no_denied_tool_calls": all(c["status"] == "SUCCESS" for c in calls),
            "evidence_tools_used": {"mura_tta_summary", "mura_gradcam_summary"} <= {c["tool"] for c in calls},
            "mentions_human_review": "human review" in text.lower(),
        },
    )
    return report

In [10]:
# 10. Sanity checks without the LLM (synthetic images, no MURA data)
def _png(seed):
    buffer = io.BytesIO()
    pixels = np.random.default_rng(seed).integers(0, 255, (256, 256), dtype=np.uint8)
    Image.fromarray(pixels).save(buffer, format="PNG")
    return buffer.getvalue()


_views = [StudyImage("v1", _png(1)), StudyImage("v2", _png(2))]


def _record(probabilities, anatomy="XR_WRIST"):
    route = guarded_study_route(case_id="synthetic", declared_anatomy=anatomy, images=_views,
                                classifier=lambda images: probabilities, threshold=THRESHOLD)
    return route, (to_policy_record("synthetic", route, probabilities) if route["classification_performed"] else None)


try:
    assert_not_official_valid(MURA_ROOT / "valid" / "XR_WRIST" / "image1.png")
    raise AssertionError("official-valid guard did not fire")
except PermissionError:
    pass

_, priority = _record([0.10, 0.90])                     # views disagree -> priority
assert agent_policy.select_agent_tools(priority, THRESHOLD)["selected_tools"] == ["tta_uncertainty", "gradcam_explainer"]
_, routine = _record([0.05, 0.10])                      # both views clearly below -> routine
assert agent_policy.select_agent_tools(routine, THRESHOLD)["selected_tools"] == []
blocked, _ = _record([0.5, 0.5], anatomy="XR_HAND")     # not wrist -> blocked before the model
assert blocked["deterministic_outcome"] == "UNSUPPORTED" and not blocked["classification_performed"]

_tools, _calls = make_gated_tools(priority, {"tta": {"status": "SUCCESS"}, "gradcam": []})
assert json.loads(_tools[0].run(study_id="synthetic"))["status"] == "SUCCESS"
assert json.loads(_tools[0].run(study_id="another-study"))["status"] == "DENIED"
_tools, _ = make_gated_tools(routine, {"tta": {}, "gradcam": []})
assert json.loads(_tools[1].run(study_id="synthetic"))["status"] == "DENIED"   # routine: tools not authorised
print("Sanity checks: PASSED")

Sanity checks: PASSED


In [11]:
# 11. Demo studies: internal validation, chosen by model output only (labels are never read)
import pandas as pd

N_SCAN = 60
manifest = pd.read_csv(VAL_MANIFEST, usecols=["study_uid", "file_name"])
manifest["image_path"] = (manifest["file_name"].astype(str).str.replace("\\", "/", regex=False)
                          .str.extract(r"(train/XR_[A-Z]+/.+)$", expand=False))
assert manifest["image_path"].notna().all(), "Unexpected file_name format in the manifest"
assert manifest["image_path"].str.startswith("train/").all(), "Internal validation must come from official train"
study_views = {uid: sorted(group["image_path"]) for uid, group in manifest.groupby("study_uid")}

scan = []
for uid in sorted(study_views)[:N_SCAN]:
    paths = [MURA_ROOT / p for p in study_views[uid]]
    probabilities = view_probabilities([StudyImage(p.name, p.read_bytes()) for p in paths])
    scan.append({"source": uid, "paths": paths, "mean": float(np.mean(probabilities)),
                 "min": min(probabilities), "max": max(probabilities)})

routine_study = min((s for s in scan if s["max"] < THRESHOLD), key=lambda s: s["mean"])
mixed = [s for s in scan if s["min"] < THRESHOLD <= s["max"]]
unclear_study = min(mixed or scan, key=lambda s: abs(s["mean"] - THRESHOLD))

hand_dir = next(p for p in sorted((MURA_ROOT / "train" / "XR_HAND").glob("*/*")) if p.is_dir())
hand_paths = sorted(p for p in hand_dir.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"})

# Anonymised IDs: MURA folder names contain the label, so they never reach the LLM.
DEMO = [
    {"case": "routine", "case_id": "DEMO_STUDY_1", "anatomy": "XR_WRIST", "paths": routine_study["paths"]},
    {"case": "unclear", "case_id": "DEMO_STUDY_2", "anatomy": "XR_WRIST", "paths": unclear_study["paths"]},
    {"case": "not_wrist", "case_id": "DEMO_STUDY_3", "anatomy": "XR_HAND", "paths": hand_paths},
]
print("Scanned:", len(scan), "| with view disagreement:", len(mixed))
for item in DEMO:
    print(f"{item['case']:10s} {item['case_id']}  views={len(item['paths'])}")

Scanned: 60 | with view disagreement: 18
routine    DEMO_STUDY_1  views=1
unclear    DEMO_STUDY_2  views=2
not_wrist  DEMO_STUDY_3  views=3


In [12]:
# 12. Run the three demo cases (the LLM runs only where the policy selects tools)
reports = []
for item in DEMO:
    report = await run_case(item["case_id"], item["anatomy"], item["paths"])
    report["demo_case"] = item["case"]
    reports.append(report)

summary = pd.DataFrame([{
    "case": r["demo_case"],
    "route": r["route"],
    "class": r.get("predicted_class"),
    "p_study": r.get("study_probability"),
    "views_disagree": r.get("view_disagreement"),
    "tools": ", ".join(r.get("selected_tools", [])) or "-",
    "LLM": r["llm_called"],
    "tokens": (r["tokens"]["input"] + r["tokens"]["output"]) if r["llm_called"] else 0,
    "cost_usd": r.get("estimated_cost_usd", 0.0),
    "seconds": r.get("duration_s"),
    "note": (r.get("llm_error") or r.get("llm_skipped") or "")[:60],
} for r in reports])
display(summary)

for r in reports:
    if r["llm_called"]:
        print(f"\n=== {r['demo_case']} ({r['case_id']}) — agent report ===\n{r['agent_report']}")
        print("Post-checks:", r["post_checks"])

,case,route,class,p_study,views_disagree,tools,LLM,tokens,cost_usd,seconds,note
0,routine,STANDARD_HUMAN_REVIEW,0.0,0.0003,False,-,False,0,0.00000,NaN,
1,unclear,RAISE_PRIORITY,0.0,0.2574,True,"tta_uncertainty, gradcam_explainer",True,2337,0.00111,2.23,
2,not_wrist,UNSUPPORTED,NaN,NaN,None,-,False,0,0.00000,NaN,



=== unclear (DEMO_STUDY_2) — agent report ===
### MURA Research Report: DEMO_STUDY_2

**Frozen Prediction:**
*   **Frozen Predicted Class:** 0 (Normal)
*   **Frozen Study Probability:** 0.2574
*   **Frozen Threshold:** 0.263568
*   **Route Reason:** Views fall on both sides of the threshold.
*   **Deterministic Route:** RAISE_PRIORITY (Priority human review).

**TTA Evidence:**
*   Test-time augmentation (TTA) results indicate a stability category of `THRESHOLD_CROSSING`.
*   Study probabilities across variants ranged from 0.2308 to 0.2707, demonstrating sensitivity to small image changes.

**Grad-CAM Evidence:**
*   Attribution maps show the model's sensitivity to different regions across views.
*   `image1.png` returned a view probability of 0.5145, while `image2.png` returned 0.0003.
*   Grad-CAM highlights areas of model sensitivity but does not localize clinical findings.

**Model Limitations:**
*   The model has low specificity at this operating point, leading to frequent flaggi

In [13]:
# 13. Save the audit report locally (the shared Drive is not modified)
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
audit_path = OUTPUT_DIR / f"agent_demo_audit_{stamp}.json"
audit = {
    "created_utc": stamp,
    "model": MODEL_ID,
    "policy_version": agent_policy.POLICY_VERSION,
    "llm_model": GEMINI_MODEL,
    "llm_runs": LLM_RUNS,
    "total_estimated_cost_usd": round(sum(r.get("estimated_cost_usd", 0.0) for r in reports), 6),
    "cases": reports,
}
text = json.dumps(audit, indent=2, ensure_ascii=False, default=str)
try:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    audit_path.write_text(text)
except OSError:                                   # no write access to the shared folder: keep a local copy
    audit_path = Path("/content/agent_outputs") / audit_path.name
    audit_path.parent.mkdir(parents=True, exist_ok=True)
    audit_path.write_text(text)
print("Saved:", audit_path)

Saved: /content/drive/MyDrive/DATA/AGENT_OUTPUTS/agent_demo_audit_20261108T090634Z.json


## What differs from Notebook 09 (semi_final 3)

| Notebook 09 | This notebook |
|---|---|
| `RESNET18_FINE_TUNING/resnet18_best.pt`, threshold 0.0601, 320 px input | Canonical Model B bundle (SHA-256 verified), threshold 0.263568, canonical 224 px preprocessing |
| Own QC/anatomy steps; route rule written twice | Canonical `guarded_study_route` before the model; its route is translated once for `agent_policy.py` |
| Evidence restored from earlier saved runs | Evidence computed for the study being run |
| Official MURA `valid` studies (`OFFICIAL_STUDY_00137`, Steps 16–18) | No official-valid paths; demo from internal validation, chosen without labels; anonymised IDs |
| ~263 code cells | 12 code cells |
| — | LLM only for priority studies, hard cap per session, tokens and cost per case |
| — | `mura_model_card` tool and model identity (bundle ID + SHA-256) in every report |

**Kept as accepted:** the trigger rule (class 1 **or** view disagreement → priority + tools), the TTA variants, Grad-CAM layer, policy v1.1, agent role/goal/backstory, Gemini model and limits.

**Open for the team:** with the canonical threshold, class 1 alone covers about 62.5 % of internal-validation studies (`frozen_primary_operating_policy.json`, `flagged_percent`), so tools run for most studies, not only unclear ones.